<a href="https://colab.research.google.com/github/anuragN2107/E-Commerce-Predictive-CLV-Churn-Dashboard/blob/main/E_Commerce_Predictive_CLV_%26_Churn_Dashboard.ipynb" target="_parent"><img src="https://colab.research.google.com/assets/colab-badge.svg" alt="Open In Colab"/></a>

#Business Problem
Customer Acquisition Costs (CAC) have increased significantly across the global e-commerce sector, making raw user acquisition less sustainable than optimization of existing customer retention.

Legacy retention operations rely on historical, descriptive reporting (e.g., standard RFM segmentation), which only identifies a customer’s value after they have already reduced transactions or churned. Without a forward-looking predictive engine, marketing teams cannot differentiate between a high-value customer with low churn risk and a high-value customer on the verge of attrition. This leads to two critical operational inefficiencies:

* **Margin Erosion:** Blanket promotional discounts are distributed to the entire customer base, wasting capital on users who would have purchased regardless.

* **Missed Revenue Recovery:** High-priority, high-value accounts at imminent risk of attrition pass unnoticed until they are entirely lost to competitors.

#Project Objective
The objective of this project is to architect and deploy an end-to-end data science and decision support system (DSS) that transitions retention workflows from descriptive to predictive and prescriptive.

Specifically, the system aims to:

* **Quantify Financial Risk:** Calculate an explicit 90-day Customer Lifetime Value (CLV) forecast and real-time churn probability score for every active customer account.

* **Isolate Priority Segments:** Provide automated, multi-tiered risk segmentation to instantly surface high-value, high-risk customer pools for localized marketing interventions.

* **Simulate Prescriptive Strategies:** Deliver a scalable financial simulation layer allowing executives to run "What-If" discount parameters and instantly calculate the expected return on marketing spend (ROMS) directly on the dashboard.

# Tool Stack
**Data Engineering & Scripting Engine:** Python 3.12

**Data Manipulation & Analysis:** Pandas, NumPy

**Statistical Machine Learning Engine:** lifetimes framework (utilizing lifetimes-native parameter optimization)

**Visualization & Analytics Interface:** Tableau (Desktop Panel Deployment Optimization)

**Source Control & Presentation Asset Layer:** GitHub Environment

#Requirements & Core Prerequisites
To execute this analytical solution, the underlying workspace must possess the following structural components:

* **Transactional Dataset Access:** A continuous transactional log containing unique client identifiers (CustomerID), chronological purchase markers (InvoiceDate), structural price indexes (UnitPrice), order scale variables (Quantity), and geographic location attributes (Country). (e.g., the UCI Machine Learning Repository Online Retail Database).

* **Environment Dependencies:** A Python runtime instance configured with isolated environment layers to mitigate package deprecation conflicts (specifically ensuring baseline positional mapping parameters for statistical package fitting functions).

* **Tableau Setup:** An environment supporting unified dashboard canvas layouts configured for responsive UI scaling (Size -> Automatic).

# Methodology & Mathematical Framework

The project is executed across three operational data lifecycle phases:
* **Phase A:Feature Engineering & RFM Transformation**
 Raw line-item transactions are aggregated into an explicit customer footprint dataframe. The data science pipeline extracts three baseline structural metrics per customer record:
  * **Frequency ($f$):** The total number of repeat purchase periods observed during the customer's lifespan.
  * **Recency ($r$):** The duration between the customer's initial transaction and their most recent transaction.
  * **Age ($T$):** The total duration between the customer's initial transaction and the final data collection snapshot index.
  * **Monetary Value ($m$):** The average transaction value calculated over the repeat purchase periods.

* **Phase B:Statistical Machine Learning Modeling**
 Instead of utilizing standard black-box classification metrics, the predictive layer applies two specialized probabilistic models optimized for non-contractual transactional environments:
  * **BG/NBD (Beta-Geometric/Negative Binomial Distribution) Model:**
    * This model maps the transactional frequency and drop-off rate of customers over time. It assumes purchase counts follow a Poisson process while a customer is active, and their unobserved lifetime follows an exponential distribution.
    * It calculates the exact Probability of Aliveness ($P(\text{Alive})$) per individual customer and outputs the expected transaction count over a forward-looking 90-day window. Churn probability is derived as the direct mathematical inverse: $1 - P(\text{Alive})$.
  * **Gamma-Gamma Conjugate Model:**
    * This model estimates the average transaction value for the upcoming periods. It assumes that monetary values vary randomly around a customer's true mean value, and that these true mean values vary across the population following a Gamma distribution.
    * Independence Constraint Met: The pipeline verifies the underlying data science prerequisite that there is no meaningful statistical correlation between transaction frequency and monetary value before fitting the Gamma-Gamma model.
  * **Customer Lifetime Value Formulation:** The expected transaction values from the Gamma-Gamma model are multiplied by the 90-day purchase expectations from the BG/NBD model to establish a strict, individual cash-flow forecast over the upcoming 90-day operating horizon.
* **Phase C:** Dashboard Integration & Prescriptive ParameterizationThe scored output dataset is joined with geographic metadata and loaded into Tableau. Inside the visual layer, parameter configurations are created to apply custom prescriptive algorithms:
  * **Simulation Rule:** If an account falls into the critical risk band ($P(\text{Churn}) \ge 0.70$), the model applies a user-selected discount factor ($\text{Discount Parameters \%}$).
  * **Prescriptive Return Metric:** The system dynamically computes Estimated Saved Revenue as $\text{Predicted CLV 90D} \times \text{Targeted Discount \%}$, allowing operators to dynamically see potential revenue recovery on a slider interface.

In [1]:
# Step 1. Install the lifetimes library
!pip install lifetimes --quiet

   ━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━ 584.2/584.2 kB 21.2 MB/s eta 0:00:00


In [7]:
# Step 2. Import core libraries
import pandas as pd
import numpy as np
import datetime as dt
import matplotlib.pyplot as plt
import seaborn as sns
import requests
import zipfile
import io
from lifetimes import BetaGeoFitter, GammaGammaFitter

In [9]:
# Step 3. Load the dataset
print("Fetching and extracting dataset from UCI repository.")
url = "https://archive.ics.uci.edu/static/public/352/online+retail.zip"

response = requests.get(url)
with zipfile.ZipFile(io.BytesIO(response.content)) as z:
    excel_file_name = [f for f in z.namelist() if f.endswith('.xlsx')][0]
    with z.open(excel_file_name) as f:
        df = pd.read_excel(f)

print(f"Data successfully loaded! Shape: {df.shape}")
df.head()

Fetching and extracting dataset from UCI repository.
Data successfully loaded! Shape: (541909, 8)


,InvoiceNo,StockCode,Description,Quantity,InvoiceDate,UnitPrice,CustomerID,Country
0,536365,85123A,WHITE HANGING HEART T-LIGHT HOLDER,6,2010-12-01 08:26:00,2.55,17850.0,United Kingdom
1,536365,71053,WHITE METAL LANTERN,6,2010-12-01 08:26:00,3.39,17850.0,United Kingdom
2,536365,84406B,CREAM CUPID HEARTS COAT HANGER,8,2010-12-01 08:26:00,2.75,17850.0,United Kingdom
3,536365,84029G,KNITTED UNION FLAG HOT WATER BOTTLE,6,2010-12-01 08:26:00,3.39,17850.0,United Kingdom
4,536365,84029E,RED WOOLLY HOTTIE WHITE HEART.,6,2010-12-01 08:26:00,3.39,17850.0,United Kingdom


In [10]:
# Step 4: Data Cleaning & Preparing
# Drop missing Customer IDs
df = df.dropna(subset=['CustomerID'])
df['CustomerID'] = df['CustomerID'].astype(int)

# Filter out cancellations
df = df[~df['InvoiceNo'].astype(str).str.startswith('C')]

# Ensure Quantity and UnitPrice are positive numbers
df = df[(df['Quantity'] > 0) & (df['UnitPrice'] > 0)]

# Handle Datetime conversions
df['InvoiceDate'] = pd.to_datetime(df['InvoiceDate'])

# Calculate Total Revenue per line item
df['TotalSales'] = df['Quantity'] * df['UnitPrice']

print(f"Data successfully cleaned! Rows remaining: {df.shape[0]}")
print(f"Unique Customers to track: {df['CustomerID'].nunique()}")

Data successfully cleaned! Rows remaining: 397884
Unique Customers to track: 4338


In [11]:
# Step 5: Calculate the RFM Matrix
from lifetimes.utils import summary_data_from_transaction_data

# Set a baseline evaluation date
snapshot_date = df['InvoiceDate'].max() + dt.timedelta(days=1)

# Compress individual txn history into a RFM dataframe
rfm = summary_data_from_transaction_data(
    df,
    customer_id_col='CustomerID',
    datetime_col='InvoiceDate',
    monetary_value_col='TotalSales',
    observation_period_end=snapshot_date
)

rfm = rfm[rfm['frequency'] > 0]
print(f"RFM Summary dataset created for {rfm.shape[0]} repeat buyers.")
rfm.head()

RFM Summary dataset created for 2790 repeat buyers.


,frequency,recency,T,monetary_value
CustomerID,,,,
12347,6.0,365.0,368.0,599.701667
12348,3.0,283.0,359.0,301.480000
12352,6.0,260.0,297.0,368.256667
12356,2.0,303.0,326.0,269.905000
12358,1.0,149.0,151.0,683.200000


In [15]:
# Step 6: Machine Learning & Predictive Modeling
from lifetimes import BetaGeoFitter, GammaGammaFitter

# Fit the BG/NBD Model using standard arguments
bgf = BetaGeoFitter()
bgf.fit(rfm['frequency'], rfm['recency'], rfm['T'])

# Calculate active likelihood and inverse churn risk
rfm['Probability_Alive'] = bgf.conditional_probability_alive(rfm['frequency'], rfm['recency'], rfm['T'])
rfm['Churn_Probability'] = 1 - rfm['Probability_Alive']

# Predict expected count of purchases
rfm['Predicted_Purchases_90D'] = bgf.conditional_expected_number_of_purchases_up_to_time(
    90, rfm['frequency'], rfm['recency'], rfm['T']
)

In [17]:
# Fit the Gamma-Gamma Model
rfm = rfm[rfm['monetary_value'] > 0]
ggf = GammaGammaFitter()
ggf.fit(rfm['frequency'], rfm['monetary_value'])

# Forecast total Customer Lifetime Value (CLV)
rfm['Predicted_CLV_90D'] = ggf.customer_lifetime_value(
    bgf,
    rfm['frequency'],
    rfm['recency'],
    rfm['T'],
    rfm['monetary_value'],
    time=3 #3 Months=90 Days
)

print("Moadel trained and predictions added successfully.")
rfm.head()

Moadel trained and predictions added successfully.


,frequency,recency,T,monetary_value,Probability_Alive,Churn_Probability,Predicted_Purchases_90D,Predicted_CLV_90D
CustomerID,,,,,,,,
12347,6.0,365.0,368.0,599.701667,0.998059,0.001941,1.493046,834.268411
12348,3.0,283.0,359.0,301.480000,0.989530,0.010470,0.938785,307.189539
12352,6.0,260.0,297.0,368.256667,0.996066,0.003934,1.749997,645.359679
12356,2.0,303.0,326.0,269.905000,0.989969,0.010031,0.805745,255.959159
12358,1.0,149.0,151.0,683.200000,0.945342,0.054658,0.958028,507.079772


In [21]:
# Step 6: Merging geographic data and exporting CSV
# Extract country registration for each CustomerID
customer_countries = df.groupby('CustomerID')['Country'].first().reset_index()

# Combine the location metadata with predictive ML table
final_data = rfm.reset_index().merge(customer_countries, on='CustomerID', how='left')

final_data.to_csv('predictive_clv_data.csv', index=False)

print("File 'predictive_clv_data.csv' has been successfully generated!")
print(f"Final Data Shape: {final_data.shape}")
final_data.head()

File 'predictive_clv_data.csv' has been successfully generated!
Final Data Shape: (2790, 10)


,CustomerID,frequency,recency,T,monetary_value,Probability_Alive,Churn_Probability,Predicted_Purchases_90D,Predicted_CLV_90D,Country
0,12347,6.0,365.0,368.0,599.701667,0.998059,0.001941,1.493046,834.268411,Iceland
1,12348,3.0,283.0,359.0,301.480000,0.989530,0.010470,0.938785,307.189539,Finland
2,12352,6.0,260.0,297.0,368.256667,0.996066,0.003934,1.749997,645.359679,Norway
3,12356,2.0,303.0,326.0,269.905000,0.989969,0.010031,0.805745,255.959159,Portugal
4,12358,1.0,149.0,151.0,683.200000,0.945342,0.054658,0.958028,507.079772,Austria


#Tableau Dashboard -

https://public.tableau.com/app/profile/anurag.srivastva/viz/EnterprisePredictiveCustomerAnalyticsIntelligenceHub/PredictiveCustomerIntelligenceHub?publish=yes

#Key Business Insights & Strategic Outcomes
Deploying this pipeline yields several critical business realizations:

* **Validation of the 80/20 Rule (Pareto Concentration):** Advanced table calculations mapping the cumulative running sum of future value show that approximately 75% to 80% of all predicted 90-day revenue is concentrated within the top 20% of the active customer base. This mathematically validates shifting marketing budgets away from broad retention efforts toward a highly concentrated VIP customer focus.

* **Precision Margin Management:** The interactive risk matrix scatter plot segments customers cleanly into quadrants. Instead of offering discounts to healthy accounts (high value, low risk) or wasting margin on lost causes (low value, high risk), the business can immediately isolate and export the specific subset of customer IDs in the High-Value, High-Risk quadrant (top-right quadrant). This maximizes retention conversion while protecting overall margin health.

* **Macro Geographic Prioritization:** Cross-filtering the geographic visual layers against the dynamic Churn Alert Banner reveals exactly which international regions are experiencing localized attrition. For example, if a specific territory presents a high average churn index but also carries a disproportionately high 90-day predicted financial runway, the executive team can immediately authorize a regional localized recovery campaign.